[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tnecniv-Teikram/hyperspectral-course-2026/blob/main/templates/project_template.ipynb)

*Running in Colab? Run the setup cell below first — it clones the course and installs what this notebook needs. Running locally? It does nothing.*

# 🛰️ <Project title>
### Space Academy 2026 · Hyperspectral course · Final graduation project

**Team:** <names> · **Theme:** <e.g. Precision agriculture> · **Version:** <date>

> **How to use this template.** Copy it next to `hsi_utils.py` (the course folder) or keep it in `templates/`,
> since the setup cell finds the helper module either way. Replace every `<…>` and every `TODO`. The example code runs
> end-to-end on the Konya agriculture scene so you can see the expected structure. Swap in your own scene,
> window and method. Keep the section order: it mirrors the evaluation rubric (`templates/evaluation_rubric.md`).

In [ ]:
# --- Setup: works both in Google Colab and in a local environment --------------------
# In Colab this clones the course repository, installs the geospatial packages that Colab
# does not ship, and moves into the course folder so the notebook finds hsi_utils and data/.
# Locally it does nothing at all — your own environment is already active.
import os
import pathlib
import subprocess
import sys

REPO = "Tnecniv-Teikram/hyperspectral-course-2026"          # change with: python tools/set_repo_url.py <owner>/<repo>

if "google.colab" in sys.modules:
    course = pathlib.Path("/content/hyperspectral-course")
    if not course.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q",
                        f"https://github.com/{REPO}.git", str(course)], check=True)
    os.chdir(course)
    sys.path.insert(0, str(course))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "h5py", "rasterio", "rioxarray", "xarray", "geopandas"], check=True)
    print("Colab ready — course at", course)
else:
    print("Local environment:", sys.executable.split("/")[-3])


## 1. Problem, user and decision
*(Copy the key points from your scoping canvas: `templates/project_scoping_template.md`.)*

- **Problem:** <who has what problem, where>
- **User and decision supported:** <e.g. irrigation authority deciding where to inspect water use>
- **Why hyperspectral:** <the spectral features you rely on, with wavelengths>
- **Success criterion:** <e.g. irrigated-field F1 ≥ 0.85 on spatially independent test blocks>

## 2. Configuration
Everything you might change lives in this one cell: scene IDs, windows, thresholds, seeds. Nothing further down
should hard-code these values.

In [ ]:
import sys, json, platform
from pathlib import Path

# find hsi_utils.py whether this notebook sits in the course folder or in templates/
for p in [Path.cwd(), Path.cwd().parent]:
    if (p / "hsi_utils.py").exists():
        sys.path.insert(0, str(p))
        break

import numpy as np
import matplotlib.pyplot as plt
import hsi_utils as hu

plt.rcParams["figure.dpi"] = 90

CONFIG = {
    "project": "my_project",                 # TODO: short name, used for output files
    "scene_id": "20250608_091605_90_4001",     # TODO: your scene(s) — find them with 16_scene_finder.ipynb
    "window": (150, 450, 250, 600),            # (row_start, row_stop, col_start, col_stop); None = full scene
    "use_cloud_mask": True,                    # inspect the mask first! (bright sabkha/salt can be flagged)
    "thresholds": {"ndvi_irrigated": 0.6},     # TODO: your parameters
    "seed": 42,
}
rng = np.random.default_rng(CONFIG["seed"])
OUT = hu.ensure_output_dir(CONFIG["project"])
print(json.dumps(CONFIG, indent=2))

## 3. Data and quality control
Open the scene, check the QA masks, and **look** at the data before any analysis.

In [ ]:
scene = hu.get_scene(CONFIG["scene_id"])      # finds the local HDF5 or downloads it (~1 GB)
print(scene.summary())

win = CONFIG["window"]
masks = scene.masks(win)
valid = scene.valid_mask(win, cloud=CONFIG["use_cloud_mask"])

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(scene.rgb(stride=3)); ax[0].set_title("Full scene (true colour, stride 3)")
if win is not None:
    r0, r1, c0, c1 = win
    ax[0].add_patch(plt.Rectangle((c0 / 3, r0 / 3), (c1 - c0) / 3, (r1 - r0) / 3, fill=False, ec="red", lw=2))
ax[1].imshow(scene.rgb(win)); ax[1].set_title("Analysis window")
ax[2].imshow(masks["cloud"], cmap="gray"); ax[2].set_title(f"Cloud flag ({masks['cloud'].mean() * 100:.1f} %)")
for a in ax: a.axis("off")
plt.tight_layout(); plt.show()
print(f"usable pixels in window: {valid.mean() * 100:.1f} %")

## 4. Preprocessing
Bad bands (water vapour, < 400 nm, > 2450 nm) are removed by `read_cube(bands="good")`.
TODO: document any extra steps (smoothing, continuum removal, normalisation) and why you need them.

In [ ]:
cube, wl = scene.read_cube(win, cloud=CONFIG["use_cloud_mask"])
print("cube:", cube.shape, f"{wl.min():.0f}–{wl.max():.0f} nm")

fig, ax = plt.subplots(figsize=(10, 4))
hu.shade_bad_regions(ax)
hu.plot_spectrum(ax, wl, np.nanmedian(cube, axis=(1, 2)), color="k", label="median spectrum")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("Reflectance"); ax.legend(); ax.grid(alpha=.3)
plt.show()

## 5. Features
TODO: the features your method uses. Name the wavelengths and give a reference for every index.

In [ ]:
band = lambda nm: cube[hu.band_index(wl, nm)]
features = {
    "NDVI": hu.nd(band(860), band(665)),
    "NDRE": hu.nd(band(790), band(720)),
    "NDMI": hu.nd(band(860), band(1650)),
}
fig, ax = plt.subplots(1, len(features), figsize=(5 * len(features), 4.5))
for a, (name, img) in zip(ax, features.items()):
    im = a.imshow(img, cmap="RdYlGn", vmin=-0.2, vmax=0.9); a.set_title(name); a.axis("off")
    plt.colorbar(im, ax=a, fraction=0.046)
plt.tight_layout(); plt.show()

## 6. Method
Always start with a **baseline** (a threshold or a simple index), then your main method. The difference between the
two is your contribution.

In [ ]:
# Baseline: a threshold on NDVI
baseline = (features["NDVI"] > CONFIG["thresholds"]["ndvi_irrigated"]).astype("float32")
baseline[~np.isfinite(features["NDVI"])] = np.nan

# TODO: main method — e.g. RF / SVM on the full spectrum (notebook 15), CNN (notebook 20),
# target detection or unmixing (notebook 14), change detection (notebook 30).
main_result = baseline.copy()   # placeholder

plt.figure(figsize=(6, 5)); plt.imshow(main_result, cmap="Greens"); plt.title("Result (placeholder = baseline)")
plt.axis("off"); plt.show()

## 7. Validation
Use **spatially independent** reference data (notebook 21). The example compares the map with ESA WorldCover
cropland in spatial blocks. That is a stand-in only: replace it with your own reference data.

In [ ]:
try:
    lab, pur = hu.load_labels(CONFIG["scene_id"], win, min_purity=9)
    ref = np.where(lab == 0, np.nan, (lab == 40).astype(float))     # 40 = WorldCover cropland
    ok = np.isfinite(ref) & np.isfinite(main_result)
    y_true, y_pred = ref[ok].astype(int), main_result[ok].astype(int)
    from sklearn.metrics import classification_report, confusion_matrix
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, target_names=["other", "cropland"], digits=3))
    print("Note: WorldCover 'cropland' includes fallow fields, so a greenness threshold cannot match it. "
          "This mismatch is exactly what your validation discussion should explain.")
except FileNotFoundError as err:
    print("No reference labels for this scene:", err)

## 8. Uncertainty and limitations
TODO: at least one of: a probability or confidence map, σ from `surface_reflectance_uncertainty`, a confidence
interval on area estimates, or a threshold sensitivity plot. Then list 2–3 limitations and how they affect the user.

In [ ]:
# Example: how sensitive is the mapped area to the threshold?
ths = np.linspace(0.3, 0.8, 11)
area_ha = [np.nansum(features["NDVI"] > t) * (scene.res_x * scene.res_y) / 1e4 for t in ths]
plt.figure(figsize=(6, 3.5)); plt.plot(ths, area_ha, "o-")
plt.xlabel("NDVI threshold"); plt.ylabel("Mapped area (ha)"); plt.grid(alpha=.3)
plt.title("Threshold sensitivity"); plt.show()

## 9. Product output
Write the result as a Cloud-Optimised GeoTIFF plus a provenance file, as your user would receive it.

In [ ]:
tif = hu.save_geotiff(OUT / f"{CONFIG['project']}_result.tif", main_result.astype("float32"),
                      scene.transform(win), scene.crs, band_names=["result"])
meta = {
    "config": CONFIG, "scene": scene.scene_id, "attribution": hu.attribution(scene.scene_id),
    "python": platform.python_version(), "numpy": np.__version__,
}
(OUT / f"{CONFIG['project']}_metadata.json").write_text(json.dumps(meta, indent=2, default=str))
print("wrote", tif)

## 10. Conclusions and next steps
- **Answer to the problem:** <one paragraph>
- **Key numbers:** <table>
- **What the user gets:** <dashboard / API / report mock-up, see notebook 31>
- **Next steps:** <more dates, EnMAP, field validation, pilot user>

## References and data credits
- Tanager STAC Data, available at www.planet.com/data/stac © 2025 Planet Labs PBC (CC-BY-4.0)
- ESA WorldCover 2021 v200 (CC-BY 4.0), if used
- <papers for your indices and methods>